# Schema-aware player merge pipeline

This notebook uses the reusable pipeline in `../src/schema_aware_player_pipeline.py`. It prints dataframe columns before feature engineering, detects available columns, merges current + previous + historical player data, and writes ML-ready outputs.

In [2]:
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from schema_aware_player_pipeline import (
    RAW_DIR,
    PROCESSED_DIR,
    read_csv,
    read_optional_csv,
    write_csv_safely,
    append_supplemental_players,
    build_player_ml_dataframe,
    build_team_ml_dataframe,
    build_coverage_report,
    build_squad_player_coverage,
    build_squad_nation_coverage,
)

print('Pipeline imported successfully')

Pipeline imported successfully


In [3]:
# Always inspect schemas before feature engineering.
current_raw = read_csv(RAW_DIR / 'players_data-2025_2026.csv', '2025-26 current season')
previous_raw = read_csv(RAW_DIR / 'players_data-2023_2024.csv', '2023-24 previous season')
historical_raw = read_csv(RAW_DIR / 'players_data-2014_2025.csv', '2014-2025 historical trends')
supplemental_raw = read_optional_csv(RAW_DIR / 'supplemental_players_2025_26.csv', '2025-26 supplemental players')
squads_raw = read_optional_csv(RAW_DIR / 'wc_squads_parsed.csv', 'parsed World Cup squads')
current_raw = append_supplemental_players(current_raw, supplemental_raw)

2025-26 current season: 2839 rows x 102 columns

========== 2025-26 current season columns (102) ==========
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 'Comp_stats_playing_time', 'Age_stats_playing_

In [4]:
player_ml = build_player_ml_dataframe(current_raw, previous_raw, historical_raw)
team_ml = build_team_ml_dataframe(player_ml)
coverage = build_coverage_report(player_ml)
squad_player_coverage = build_squad_player_coverage(squads_raw, player_ml)
squad_nation_coverage = build_squad_nation_coverage(squad_player_coverage)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
write_csv_safely(player_ml, PROCESSED_DIR / 'player_ml_features.csv')
write_csv_safely(team_ml, PROCESSED_DIR / 'team_ml_features.csv')
write_csv_safely(coverage, PROCESSED_DIR / 'coverage_report.csv')
if squad_player_coverage is not None:
    write_csv_safely(squad_player_coverage, PROCESSED_DIR / 'squad_player_coverage.csv')
if squad_nation_coverage is not None:
    write_csv_safely(squad_nation_coverage, PROCESSED_DIR / 'squad_nation_coverage.csv')

print('player_ml:', player_ml.shape)
print('team_ml:', team_ml.shape)
print('coverage:', coverage.shape)
if squad_player_coverage is not None:
    print('squad_player_coverage:', squad_player_coverage.shape)
if squad_nation_coverage is not None:
    print('squad_nation_coverage:', squad_nation_coverage.shape)

player_ml: (3552, 72)
team_ml: (48, 67)
coverage: (48, 10)
squad_player_coverage: (1085, 11)
squad_nation_coverage: (36, 6)


In [5]:
coverage.sort_values(['coverage_level', 'player_count']).head(15)

,nation_code,nation_name,player_count,current_players,previous_players,historical_minutes,supplemental_players,verified_supplemental_players,estimated_supplemental_players,coverage_level
42,AUS,Australia,5.0,4.0,1.0,4811.0,0.0,0.0,0.0,low
20,ECU,Ecuador,10.0,9.0,5.0,17286.0,0.0,0.0,0.0,low
24,CAN,Canada,11.0,9.0,5.0,11260.0,0.0,0.0,0.0,low
36,DZA,Algeria,0.0,0.0,0.0,0.0,0.0,0.0,0.0,missing
41,SAU,Saudi Arabia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,missing
46,IRQ,Iraq,0.0,0.0,0.0,0.0,0.0,0.0,0.0,missing
26,CUW,Curacao,25.0,25.0,0.0,0.0,25.0,0.0,25.0,usable
27,HAI,Haiti,26.0,25.0,2.0,18621.0,21.0,0.0,21.0,usable
31,EGY,Egypt,26.0,25.0,4.0,21033.0,21.0,0.0,21.0,usable
47,NZL,New Zealand,26.0,25.0,2.0,10843.0,24.0,0.0,24.0,usable


In [6]:
team_ml[['nation_code', 'nation_name', 'player_count', 'has_player_coverage', 'weighted_chance_form_sum', 'weighted_goal_form_sum']].head(20)

,nation_code,nation_name,player_count,has_player_coverage,weighted_chance_form_sum,weighted_goal_form_sum
1,FRA,France,458.0,True,16.282155,37.849043
3,GER,Germany,327.0,True,14.278558,28.132982
2,ESP,Spain,569.0,True,13.294127,36.226397
0,ENG,England,288.0,True,10.237875,26.646596
17,BRA,Brazil,143.0,True,6.457530,13.133160
16,ARG,Argentina,119.0,True,4.768531,11.944616
4,POR,Portugal,92.0,True,3.835059,15.347550
6,BEL,Belgium,78.0,True,3.670057,8.148979
5,NED,Netherlands,96.0,True,3.601518,8.709566
19,COL,Colombia,45.0,True,3.085292,5.645603
